# CO_full_evaluation
**Distribution-Free Recalibration of Tail Quantile Forecasts under Temporal Dependence**

Pele, D.T., Lessmann, S., Härdle, W.K. (2026)

Main evaluation pipeline: 9 forecasters × 24 assets × 4 alpha levels.
Conformal VaR recalibration with Kupiec, Basel TL, and Quantile Score backtests.

**Repository:** [QuantLet/Conformal_Oracle](https://github.com/QuantLet/Conformal_Oracle)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Data directory — adjust for your environment
# ---------------------------------------------------------------------------
DATA_DIR = Path('../cfp_ijf_data')
if not DATA_DIR.exists():
    DATA_DIR = Path('/Users/danielpele/Documents/2026 CFP LLM VaR/cfp_ijf_data')

assert DATA_DIR.exists(), f'Data directory not found: {DATA_DIR}'
print(f'DATA_DIR = {DATA_DIR.resolve()}')

# ---------------------------------------------------------------------------
# Assets (24)
# ---------------------------------------------------------------------------
ASSETS = [
    'ASX200', 'AUDUSD', 'BOVESPA', 'BTC', 'CACT', 'CBU0', 'DJCI', 'ETH',
    'EURUSD', 'FTSE100', 'GBPUSD', 'GDAXI', 'GOLD', 'HSI', 'IBGL', 'ICLN',
    'NATGAS', 'NIFTY', 'NIKKEI', 'SP500', 'STOXX', 'TLT', 'USDJPY', 'WTI',
]

# ---------------------------------------------------------------------------
# Models — (subdirectory, filename pattern)
# ---------------------------------------------------------------------------
MODEL_DIRS = {
    'Chronos-Small': ('chronos_small', '{asset}.parquet'),
    'Chronos-Mini':  ('chronos_mini',  '{asset}.parquet'),
    'TimesFM-2.5':   ('timesfm25',     '{asset}.parquet'),
    'Moirai-2.0':    ('moirai2',       '{asset}.parquet'),
    'Lag-Llama':     ('lagllama',      '{asset}.parquet'),
}
BENCH_DIRS = {
    'GJR-GARCH': ('benchmarks', '{asset}_gjr_garch.parquet'),
    'GARCH-N':   ('benchmarks', '{asset}_garch_n.parquet'),
    'Hist-Sim':  ('benchmarks', '{asset}_hs.parquet'),
    'EWMA':      ('benchmarks', '{asset}_ewma.parquet'),
}
ALL_MODELS = {**MODEL_DIRS, **BENCH_DIRS}

# ---------------------------------------------------------------------------
# Evaluation parameters
# ---------------------------------------------------------------------------
ALPHAS = [0.01, 0.025, 0.05, 0.10]
F_CAL  = 0.70

print(f'Models: {len(ALL_MODELS)}  |  Assets: {len(ASSETS)}  |  Alphas: {len(ALPHAS)}')
print(f'Calibration fraction: {F_CAL}')

## Data Loading

In [ ]:
def load_pair(model, asset, alpha):
    """Load returns and VaR forecasts, align by date.

    Returns
    -------
    returns : np.ndarray   (aligned daily log-returns)
    var_raw : np.ndarray   (raw VaR forecasts, negative numbers)
    """
    subdir, pattern = ALL_MODELS[model]
    fname = pattern.format(asset=asset)

    ret = pd.read_csv(
        DATA_DIR / 'returns' / f'{asset}.csv',
        index_col=0, parse_dates=True,
    )
    fcast = pd.read_parquet(DATA_DIR / subdir / fname)

    col = f'VaR_{alpha}'
    if col not in fcast.columns:
        raise KeyError(f'{col} not in {subdir}/{fname}')

    fcast = fcast.dropna(subset=[col])
    common = ret.index.intersection(fcast.index).sort_values()

    if len(common) < 50:
        raise ValueError(f'Only {len(common)} overlapping dates for {model}/{asset}')

    return ret.loc[common, 'log_return'].values, fcast.loc[common, col].values


# Quick sanity check
r, v = load_pair('Chronos-Small', 'SP500', 0.01)
print(f'Chronos-Small / SP500 / alpha=0.01  ->  {len(r)} obs,  '
      f'mean return {r.mean():.5f},  mean VaR {v.mean():.5f}')

## Conformal Calibration

In [ ]:
def conformal_backtest(returns, var_raw, alpha, f_cal=0.70):
    """One-sided conformal VaR correction + backtests.

    Parameters
    ----------
    returns : array, daily log-returns (aligned with var_raw)
    var_raw : array, raw VaR forecasts (negative numbers)
    alpha   : float, nominal tail probability
    f_cal   : float, fraction used for calibration

    Returns
    -------
    dict with keys: n_cal, n_test, qV, pihat_raw, pihat_cp,
         viol_raw, viol_cp, p_kup_raw, p_kup_cp, p_cc_raw, p_cc_cp,
         TL_raw, TL_cp, QS_raw, QS_cp, VaR_width
    """
    n = len(returns)
    n_cal = int(n * f_cal)
    n_test = n - n_cal

    r_cal, v_cal = returns[:n_cal], var_raw[:n_cal]
    r_test, v_test = returns[n_cal:], var_raw[n_cal:]

    # ---- Nonconformity scores (one-sided) ----
    scores = v_cal - r_cal
    qV = np.quantile(scores, 1 - alpha, method='higher')

    # ---- Corrected VaR ----
    var_cp = v_test - qV

    # ---- Violations ----
    viol_raw = int(np.sum(r_test < v_test))
    viol_cp  = int(np.sum(r_test < var_cp))
    pihat_raw = viol_raw / n_test
    pihat_cp  = viol_cp / n_test

    # ---- Kupiec POF LR test ----
    def kupiec_pval(n, v, a):
        if v == 0:
            lr = -2.0 * n * np.log(1.0 - a)
        elif v == n:
            lr = -2.0 * n * np.log(a)
        else:
            pihat = v / n
            lr = -2.0 * (
                v * np.log(a / pihat) +
                (n - v) * np.log((1.0 - a) / (1.0 - pihat))
            )
        return 1.0 - stats.chi2.cdf(abs(lr), 1)

    p_kup_raw = kupiec_pval(n_test, viol_raw, alpha)
    p_kup_cp  = kupiec_pval(n_test, viol_cp,  alpha)

    # ---- Christoffersen independence test ----
    def cc_pval(violations_bool):
        v = violations_bool.astype(int)
        n00 = int(np.sum((v[:-1] == 0) & (v[1:] == 0)))
        n01 = int(np.sum((v[:-1] == 0) & (v[1:] == 1)))
        n10 = int(np.sum((v[:-1] == 1) & (v[1:] == 0)))
        n11 = int(np.sum((v[:-1] == 1) & (v[1:] == 1)))

        if (n00 + n01) == 0 or (n10 + n11) == 0 or (n01 + n11) == 0:
            return 1.0

        pi01 = n01 / (n00 + n01)
        pi11 = n11 / (n10 + n11)
        pi   = (n01 + n11) / (n00 + n01 + n10 + n11)

        if pi01 in (0, 1) or pi11 in (0, 1) or pi in (0, 1):
            return 1.0

        lr_ind = -2.0 * (
            np.log((1 - pi) ** (n00 + n10) * pi ** (n01 + n11))
            - np.log(
                (1 - pi01) ** n00 * pi01 ** n01
                * (1 - pi11) ** n10 * pi11 ** n11
            )
        )
        return 1.0 - stats.chi2.cdf(abs(lr_ind), 1)

    p_cc_raw = cc_pval(r_test < v_test)
    p_cc_cp  = cc_pval(r_test < var_cp)

    # ---- Basel Traffic Light (scaled to test length) ----
    def basel_tl(n_viol, n_days):
        scaled = n_viol * 250.0 / n_days
        if scaled <= 4:
            return 'Green'
        elif scaled <= 9:
            return 'Yellow'
        else:
            return 'Red'

    tl_raw = basel_tl(viol_raw, n_test)
    tl_cp  = basel_tl(viol_cp,  n_test)

    # ---- Quantile Score ----
    def quantile_score(r, v, a):
        indicator = (r < v).astype(float)
        return float(np.mean((a - indicator) * (r - v)))

    qs_raw = quantile_score(r_test, v_test, alpha)
    qs_cp  = quantile_score(r_test, var_cp,  alpha)

    # ---- VaR width (mean absolute corrected VaR) ----
    var_width = float(np.mean(np.abs(var_cp)))

    return {
        'n_cal': n_cal, 'n_test': n_test, 'qV': float(qV),
        'pihat_raw': pihat_raw, 'pihat_cp': pihat_cp,
        'viol_raw': viol_raw, 'viol_cp': viol_cp,
        'p_kup_raw': p_kup_raw, 'p_kup_cp': p_kup_cp,
        'p_cc_raw': p_cc_raw, 'p_cc_cp': p_cc_cp,
        'TL_raw': tl_raw, 'TL_cp': tl_cp,
        'QS_raw': qs_raw, 'QS_cp': qs_cp,
        'VaR_width': var_width,
    }

## Run Full Evaluation

In [ ]:
rows = []
errors = []

for model in ALL_MODELS:
    for asset in ASSETS:
        for alpha in ALPHAS:
            try:
                r, v = load_pair(model, asset, alpha)
                res = conformal_backtest(r, v, alpha, F_CAL)
                res.update({'model': model, 'symbol': asset, 'alpha': alpha})
                rows.append(res)
            except Exception as e:
                errors.append(f'{model}/{asset}/{alpha}: {e}')

df = pd.DataFrame(rows)
print(f'Computed: {len(df)} combinations '
      f'({df["model"].nunique()} models x '
      f'{df["symbol"].nunique()} assets x '
      f'{df["alpha"].nunique()} alphas)')
if errors:
    print(f'\nErrors: {len(errors)}')
    for e in errors[:10]:
        print(f'  {e}')

## Results: Raw vs Corrected Violation Rates (alpha = 0.01)

In [ ]:
d01 = df[df['alpha'] == 0.01].copy()

summary = d01.groupby('model').agg(
    mean_pihat_raw=('pihat_raw', 'mean'),
    mean_pihat_cp=('pihat_cp', 'mean'),
    mean_qV=('qV', 'mean'),
    green_raw=('TL_raw', lambda x: (x == 'Green').sum()),
    green_cp=('TL_cp', lambda x: (x == 'Green').sum()),
    red_cp=('TL_cp', lambda x: (x == 'Red').sum()),
    kupiec_pass_raw=('p_kup_raw', lambda x: (x > 0.05).sum()),
    kupiec_pass_cp=('p_kup_cp', lambda x: (x > 0.05).sum()),
    mean_QS_raw=('QS_raw', 'mean'),
    mean_QS_cp=('QS_cp', 'mean'),
).round(4)

# Sort by mean_qV descending (most miscalibrated first)
summary = summary.sort_values('mean_qV', ascending=False)

print('=== Master Summary: alpha = 0.01, 24 assets ===')
print(f'Total Green (corrected): {d01["TL_cp"].eq("Green").sum()}/{len(d01)}')
print(f'Total Red   (corrected): {d01["TL_cp"].eq("Red").sum()}/{len(d01)}')
print()
print(summary.to_string())

## qV Ranking by Model

In [ ]:
qv_rank = d01.groupby('model').agg(
    mean_qV=('qV', 'mean'),
    std_qV=('qV', 'std'),
    min_qV=('qV', 'min'),
    max_qV=('qV', 'max'),
    sign_pos=('qV', lambda x: (x > 0).sum()),
    sign_neg=('qV', lambda x: (x < 0).sum()),
).sort_values('mean_qV', ascending=False).round(4)

print('=== qV Ranking (alpha = 0.01) ===')
print('Positive qV -> under-coverage (needs widening)')
print('Negative qV -> over-coverage (conservative)')
print()
print(qv_rank.to_string())

## Multi-Quantile Evaluation

In [ ]:
multi = df.groupby(['alpha', 'model']).agg(
    mean_pihat_cp=('pihat_cp', 'mean'),
    green=('TL_cp', lambda x: (x == 'Green').sum()),
    kupiec_pass=('p_kup_cp', lambda x: (x > 0.05).sum()),
    mean_QS=('QS_cp', 'mean'),
).round(4)

print('=== Multi-Quantile Summary ===')
for alpha in ALPHAS:
    print(f'\n--- alpha = {alpha} ---')
    sub = multi.loc[alpha].sort_values('mean_QS')
    print(sub.to_string())

## Save Results

In [ ]:
out_dir = Path('results')
out_dir.mkdir(exist_ok=True)
df.to_csv(out_dir / 'all_results.csv', index=False)
summary.to_csv(out_dir / 'master_summary.csv')
qv_rank.to_csv(out_dir / 'qv_ranking.csv')

print('Saved to results/')
print()
print('=' * 60)
print('KEY RESULT: Conformal VaR Recalibration (alpha = 0.01)')
print('=' * 60)
n_green = d01['TL_cp'].eq('Green').sum()
n_red   = d01['TL_cp'].eq('Red').sum()
n_total = len(d01)
print(f'Basel Green Zone: {n_green}/{n_total} ({100 * n_green / n_total:.1f}%)')
print(f'Basel Red Zone:   {n_red}/{n_total}')
print(f'Kupiec pass (5%): {(d01["p_kup_cp"] > 0.05).sum()}/{n_total}')